# Subida de curvas · entrada y salida

**Para el front.** Esto es todo el contrato de la pantalla de subida: lo que la web manda y lo
que recibe de vuelta.

## Resumen en tres lineas

| | |
|---|---|
| **manda** | un fichero. Y otro, si hay generacion |
| **recibe** | un JSON con lo que el servidor ha leido de ese fichero |
| **pregunta** | nada |

No hay campo de consumo anual, ni de kWp, ni de unidad. **Todo eso ya esta dentro del
fichero**, asi que se lee en vez de preguntarse. Pedirlo aparte seria pedir dos veces el mismo
dato y arriesgarse a que las dos respuestas no coincidan — y cuando no coinciden, no hay forma
de saber cual de las dos es la buena.

Lo que la web SI tiene que hacer es **enseñar lo que ha leido** antes de dejar continuar. Ese
es el sitio donde el usuario detecta que ha subido el fichero del año pasado, o el de la otra
nave.

In [1]:
import sys, os, json, subprocess
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "production" / "app"))
os.environ.setdefault("TFM_EMAIL", "acjg.sgs@outlook.com")
pd.set_option("display.width", 200, "display.max_colwidth", 70)

---
# 1 · ENTRADA

Dos ficheros. Es literalmente todo.

In [2]:
ENTRADA = {
    "consumo":    "docs/plantillas/plantilla_consumo.csv",
    "generacion": "docs/plantillas/plantilla_generacion.csv",
}

for k, v in ENTRADA.items():
    p = REPO / v
    print(f"  {k:12s} {p.name:30s} {p.stat().st_size:>9,} bytes".replace(",", "."))

  consumo      plantilla_consumo.csv            179.723 bytes
  generacion   plantilla_generacion.csv         172.303 bytes


### Formatos que se aceptan

El servidor reconoce los tres solo, sin que la web tenga que decírselo:

| formato | pinta | de dónde suele venir |
|---|---|---|
| **ancho** | una fila por día, columnas `H1..H24` (o `H25`) | Excel de distribuidora |
| **largo** | `fecha`, `hora`, `valor` | sistemas de medida |
| **largo** | una columna de fecha-hora y otra de valor | dataloggers |

Acepta `.csv`, `.xlsx`, `.xls` y `.xlsm`. En Excel **no hace falta que la curva esté en la
primera hoja**: se prueban todas y se coge la que parezca una curva, así que una portada o una
hoja de instrucciones delante no molesta.

### Lo único que la web debería vigilar antes de subir

**Un año completo.** Con menos, los meses que falten se rellenan con la media de los demás, y
en una fábrica con parón de agosto eso deforma el resultado. Se puede subir más de un año: se
promedian.

**La unidad, en el nombre de la columna** — `consumo_kwh`, `generacion_kwh`. Si no está, el
servidor avisa y **no adivina**: 40 por hora es 40 kWh en una pyme y 40 MWh en una fábrica, y
por la magnitud no se distinguen. En formato ancho las columnas se llaman `H1..H24` y no hay
dónde escribirla, así que ahí **hay que preguntarla** y pasarla como `--unidad`.

Las plantillas de `docs/plantillas/` traen el año completo y la unidad puesta. Conviene
ofrecerlas para descargar en la propia pantalla de subida.

---
# 2 · LA LLAMADA

In [3]:
ORDEN = [sys.executable, "production/app/cargar_curvas.py", "--json",
         "--consumo",    ENTRADA["consumo"],
         "--generacion", ENTRADA["generacion"],
         "--code-consumo", "CURVA-CON", "--code-generacion", "CURVA-GEN"]

print("  " + " ".join(ORDEN[1:]))

  production/app/cargar_curvas.py --json --consumo docs/plantillas/plantilla_consumo.csv --generacion docs/plantillas/plantilla_generacion.csv --code-consumo CURVA-CON --code-generacion CURVA-GEN


---
# 3 · SALIDA

Un JSON por curva. **La forma es siempre la misma**, haya ido bien o mal: `ok` dice cuál de
los dos casos es. Un front que tiene que distinguir dos formas de respuesta acaba con dos
caminos y uno de los dos sin probar nunca.

In [4]:
r = subprocess.run(ORDEN, cwd=REPO, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
SALIDA = json.loads(r.stdout)
print(json.dumps(SALIDA, indent=2, ensure_ascii=False))

[
  {
    "tipo": "consumo",
    "ok": true,
    "code": "CURVA-CON",
    "id": 7,
    "problemas": [],
    "curva": {
      "desde": "2027-01-01",
      "hasta": "2027-12-31",
      "dias": 365,
      "horas": 8760,
      "anos": 1.0,
      "anual_mwh": 374.2,
      "pico_kw": 75.7,
      "media_kw": 42.7,
      "nulos": 0,
      "negativos": 0
    }
  },
  {
    "tipo": "generacion",
    "ok": true,
    "code": "CURVA-GEN",
    "id": 10,
    "problemas": [],
    "curva": {
      "desde": "2027-01-01",
      "hasta": "2027-12-31",
      "dias": 365,
      "horas": 8760,
      "anos": 1.0,
      "anual_mwh": 177.2,
      "pico_kw": 207.7,
      "media_kw": 20.2,
      "nulos": 0,
      "negativos": 0,
      "kwp_equivalentes": 110.8
    }
  }
]


### Los campos

| campo | qué es | para qué lo usa la web |
|---|---|---|
| `ok` | si se ha guardado | habilitar o no el botón de continuar |
| `code` | el identificador de la instalación | referenciarla al lanzar el estudio |
| `id` | la clave en la base, o `null` | — |
| `problemas` | lista de motivos, vacía si todo bien | enseñarlos tal cual: están escritos para el usuario |
| `curva.desde` / `.hasta` | el periodo que trae el fichero | **enseñarlo**: aquí se ve si subió el año que no era |
| `curva.dias` / `.horas` | cuánto hay | 365 días y 8.760 horas es lo esperado |
| `curva.anos` | años que cubre | menos de 1 es motivo de rechazo |
| `curva.anual_mwh` | el consumo o la generación anual **leídos del fichero** | **enseñarlo grande**: es el número que el usuario reconoce |
| `curva.pico_kw` | la hora más alta | contrastar con la potencia contratada |
| `curva.media_kw` | la media horaria | — |
| `curva.nulos` / `.negativos` | huecos y valores imposibles | avisar si no son cero |
| `curva.kwp_equivalentes` | solo en generación: la potencia que corresponde a esa energía | **enseñarlo**: si no se parece a lo que el usuario tiene instalado, algo falla |

`kwp_equivalentes` sale de dividir la energía del fichero entre 1.600 kWh/kWp, que es el
rendimiento de referencia del motor. No pretende ser la placa de características: es un
contraste. Si alguien tiene 250 kWp instalados y aquí salen 110, o el fichero es de otra
instalación o está en otra unidad.

In [5]:
# la tarjeta que la web enseña despues de subir
for s in SALIDA:
    c = s["curva"]
    print(f"\n  {s['tipo'].upper():12s} {'OK' if s['ok'] else 'RECHAZADA'}")
    print(f"    periodo         {c['desde']} -> {c['hasta']}  ({c['dias']} dias)")
    print(f"    anual           {c['anual_mwh']:,.1f} MWh".replace(",", "."))
    print(f"    pico            {c['pico_kw']:,.1f} kW".replace(",", "."))
    if "kwp_equivalentes" in c:
        print(f"    equivale a      {c['kwp_equivalentes']:,.1f} kWp".replace(",", "."))
    if c["nulos"] or c["negativos"]:
        print(f"    OJO             {c['nulos']} nulos · {c['negativos']} negativos")


  CONSUMO      OK
    periodo         2027-01-01 -> 2027-12-31  (365 dias)
    anual           374.2 MWh
    pico            75.7 kW

  GENERACION   OK
    periodo         2027-01-01 -> 2027-12-31  (365 dias)
    anual           177.2 MWh
    pico            207.7 kW
    equivale a      110.8 kWp


---
# 4 · CUANDO EL FICHERO ESTÁ MAL

Lo mismo, subiendo el fichero de **consumo** en la casilla de **generación**. Es el error más
fácil de cometer y el más difícil de ver después.

In [6]:
r = subprocess.run(
    [sys.executable, "production/app/cargar_curvas.py", "--json",
     "--generacion", ENTRADA["consumo"], "--code-generacion", "NO-DEBE"],
    cwd=REPO, capture_output=True, text=True, encoding="utf-8", errors="replace")
MALO = json.loads(r.stdout)
print(json.dumps(MALO, indent=2, ensure_ascii=False))

[
  {
    "tipo": "generacion",
    "ok": false,
    "code": "NO-DEBE",
    "id": null,
    "problemas": [
      "genera 0.819 de su media entre las 0:00 y las 4:00, y una fotovoltaica da CERO: ¿es este el fichero de generacion?"
    ],
    "curva": {
      "desde": "2027-01-01",
      "hasta": "2027-12-31",
      "dias": 365,
      "horas": 8760,
      "anos": 1.0,
      "anual_mwh": 374.2,
      "pico_kw": 75.7,
      "media_kw": 42.7,
      "nulos": 0,
      "negativos": 0
    }
  }
]


`ok: false`, `id: null` y el motivo en `problemas`, escrito para que se le pueda enseñar al
usuario tal cual. **No se ha guardado nada**: la transacción se deshace entera.

Esto no es una comprobación de adorno. En las pruebas de este proyecto una instalación acabó
cargada exactamente así, y el estudio corrió, guardó resultados y devolvió un VAN con dos
decimales — describiendo un emplazamiento que no existe. Con el perfil corregido, la
recomendación pasó de *"no sale a ninguna talla"* a *"sale, y en el 100 % de los escenarios"*.

Se puede saltar con `--forzar`, pero entonces es una decisión de alguien, no un descuido.

---
# 5 · LA COMPROBACIÓN QUE VALE LA PENA ENSEÑAR

El día medio de las dos curvas, una encima de la otra. Es la forma más rápida de ver que no se
han cruzado los ficheros: **el consumo no baja a cero de noche y la generación sí**.

In [7]:
from caso import conexion
con = conexion()
e = {"e": os.environ["TFM_EMAIL"]}

d = pd.read_sql("""
    SELECT 'consumo' AS curva, s.hour AS hora, avg(s.value_pu) AS pu
      FROM app_consump_shape s JOIN app_consump_inst i USING (consump_id)
      JOIN app_user u USING (user_id)
     WHERE u.email = %(e)s AND i.code = 'CURVA-CON' GROUP BY s.hour
    UNION ALL
    SELECT 'generacion', s.hour, avg(s.value_pu)
      FROM app_gen_shape s JOIN app_gen_inst i USING (gen_id)
      JOIN app_user u USING (user_id)
     WHERE u.email = %(e)s AND i.code = 'CURVA-GEN' GROUP BY s.hour
     ORDER BY 1, 2""", con, params=e)
t = d.pivot(index="hora", columns="curva", values="pu")

print("  hora   consumo                    generacion")
for hh, fila in t.iterrows():
    print(f"   {int(hh):02d}h   {fila.consumo:4.2f} {'#'*int(fila.consumo*16):<22s}"
          f"  {fila.generacion:4.2f} {'#'*int(fila.generacion*16)}")
print(f"\n  de 0:00 a 4:00 -> consumo {t.loc[0:4].consumo.mean():.2f} · "
      f"generacion {t.loc[0:4].generacion.mean():.2f}")

  hora   consumo                    generacion
   00h   0.73 ###########             0.00 
   01h   0.73 ###########             0.00 
   02h   0.73 ###########             0.00 
   03h   0.73 ###########             0.00 
   04h   0.73 ###########             0.00 
   05h   0.75 ###########             0.00 
   06h   0.78 ############            0.00 
   07h   0.83 #############           0.02 
   08h   0.92 ##############          0.10 #
   09h   1.01 ################        0.37 #####
   10h   1.10 #################       1.05 ################
   11h   1.13 ##################      2.28 ####################################
   12h   1.10 #################       3.79 ############################################################
   13h   1.03 ################        4.82 #############################################################################
   14h   0.96 ###############         4.69 ###########################################################################
   15h   0.93 #########

C:\Users\torgi\AppData\Local\Temp\ipykernel_55548\1686068638.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  d = pd.read_sql("""


In [8]:
con.close()
print("  fin")

  fin


---
## Para llevarse

**La pantalla de subida tiene dos botones de fichero y ningún campo más.**

**Después de subir, enseñar la tarjeta**: periodo, MWh anuales, pico y —en generación— los kWp
equivalentes. Es donde el usuario reconoce sus propios datos, o no.

**Si `ok` es `false`, enseñar `problemas` tal cual** y no dejar continuar. Los mensajes están
escritos para leerse, no para depurar.

**Ofrecer las plantillas** de `docs/plantillas/` en la misma pantalla. Traen el año completo,
la unidad en el encabezado y los dos días raros del cambio de hora ya resueltos como ejemplo.